# Rede Neural ADALINE — Classificação de Sinais para Ajuste de Válvulas A/B
**CEFET-MG Campus VIII – Varginha**  
**Disciplina:** Laboratório de Inteligência Artificial  
**Professor:** Lázaro Eduardo da Silva  

---
## Contexto do Problema
Um sistema envia sinais codificados de 4 grandezas $\{x_1, x_2, x_3, x_4\}$ para ajuste de duas válvulas (**A** e **B**).
Uma rede **ADALINE** é treinada via **Regra Delta (LMS)** com taxa $\eta = 0.0025$ e precisão $\epsilon = 10^{-6}$ para indicar ao comutador se o sinal é para:
- **Válvula A** (convenção: `-1`)
- **Válvula B** (convenção: `+1`)

Os dados de treinamento e teste são carregados diretamente dos arquivos `adaline_treinamento.csv` e `adaline_teste.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Implementação da Classe ADALINE

In [ ]:
class Adaline:
    def __init__(self, eta=0.0025, precisao=1e-6, max_epocas=50000, bias_input=-1):
        self.eta = eta
        self.precisao = precisao
        self.max_epocas = max_epocas
        self.bias_input = bias_input
        self.w = None
        self.w_inicial = None
        self.historico_eqm = []
        self.epocas_executadas = 0

    def calcular_eqm(self, X, d):
        u = np.dot(X, self.w)
        return np.mean((d - u) ** 2)

    def treinar(self, X, d, seed=None):
        if seed is not None:
            np.random.seed(seed)
        
        n_amostras, n_atributos = X.shape
        self.w = np.random.rand(n_atributos)
        self.w_inicial = self.w.copy()
        self.historico_eqm = []
        self.epocas_executadas = 0
        
        eqm_anterior = self.calcular_eqm(X, d)
        self.historico_eqm.append(eqm_anterior)
        
        for epoca in range(self.max_epocas):
            for i in range(n_amostras):
                u = np.dot(self.w, X[i])
                erro = d[i] - u
                self.w += self.eta * erro * X[i]
            
            eqm_atual = self.calcular_eqm(X, d)
            self.historico_eqm.append(eqm_atual)
            self.epocas_executadas += 1
            
            if abs(eqm_atual - eqm_anterior) < self.precisao:
                break
                
            eqm_anterior = eqm_atual
            
        return self

    def predizer(self, X):
        u = np.dot(X, self.w)
        return np.where(u >= 0, 1.0, -1.0)

## Carregando o Conjunto de Treinamento (`adaline_treinamento.csv`)

In [ ]:
caminho_treino = '../atividades/perceptron_adaline/adaline_treinamento.csv'
if not os.path.exists(caminho_treino):
    caminho_treino = 'atividades/perceptron_adaline/adaline_treinamento.csv'

df_treino = pd.read_csv(caminho_treino)
print("Primeiras linhas do adaline_treinamento.csv:")
display(df_treino.head())

X_raw = df_treino[['x1', 'x2', 'x3', 'x4']].values
d_treino = df_treino['d'].values
X_treino = np.hstack((np.full((X_raw.shape[0], 1), -1.0), X_raw))

print(f"Conjunto de Treinamento ADALINE: {X_treino.shape[0]} amostras carregadas do CSV.")

## Questões 1 e 2 — Execução e Registro dos 5 Treinamentos

In [ ]:
seeds = [1, 2, 3, 4, 5]
modelos = []
tabela_q2 = []

for idx, seed in enumerate(seeds, 1):
    ada = Adaline(eta=0.0025, precisao=1e-6, max_epocas=50000)
    ada.treinar(X_treino, d_treino, seed=seed)
    modelos.append(ada)
    
    tabela_q2.append({
        'Treinamento': f'{idx}º (T{idx})',
        'w0 (Inicial)': f'{ada.w_inicial[0]:.4f}',
        'w1 (Inicial)': f'{ada.w_inicial[1]:.4f}',
        'w2 (Inicial)': f'{ada.w_inicial[2]:.4f}',
        'w3 (Inicial)': f'{ada.w_inicial[3]:.4f}',
        'w4 (Inicial)': f'{ada.w_inicial[4]:.4f}',
        'w0 (Final)': f'{ada.w[0]:.4f}',
        'w1 (Final)': f'{ada.w[1]:.4f}',
        'w2 (Final)': f'{ada.w[2]:.4f}',
        'w3 (Final)': f'{ada.w[3]:.4f}',
        'w4 (Final)': f'{ada.w[4]:.4f}',
        'Épocas': ada.epocas_executadas,
        'EQM Final': f'{ada.historico_eqm[-1]:.6f}'
    })

df_q2 = pd.DataFrame(tabela_q2)
display(df_q2.style.set_caption("Resultados dos 5 Treinamentos do ADALINE"))

## Questão 3 — Gráfico do Erro Quadrático Médio (EQM) para T1 e T2

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(modelos[0].historico_eqm, label=f'Treinamento 1 (T1) — {modelos[0].epocas_executadas} épocas', color='#00d4ff', linewidth=2)
plt.plot(modelos[1].historico_eqm, label=f'Treinamento 2 (T2) — {modelos[1].epocas_executadas} épocas', color='#ff6b9d', linestyle='--', linewidth=2)

plt.title('Evolução do Erro Quadrático Médio (EQM) por Época — T1 e T2')
plt.xlabel('Época')
plt.ylabel('Erro Quadrático Médio (EQM)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Questão 4 — Classificação das 15 Amostras de Sinais (`adaline_teste.csv`)

In [ ]:
caminho_teste = '../atividades/perceptron_adaline/adaline_teste.csv'
if not os.path.exists(caminho_teste):
    caminho_teste = 'atividades/perceptron_adaline/adaline_teste.csv'

df_teste = pd.read_csv(caminho_teste)
X_teste_raw = df_teste[['x1', 'x2', 'x3', 'x4']].values
X_teste = np.hstack((np.full((X_teste_raw.shape[0], 1), -1.0), X_teste_raw))

tabela_q4 = []
for i in range(len(X_teste_raw)):
    linha = {
        'Amostra': int(df_teste.iloc[i]['Amostra']),
        'x1': X_teste_raw[i, 0],
        'x2': X_teste_raw[i, 1],
        'x3': X_teste_raw[i, 2],
        'x4': X_teste_raw[i, 3]
    }
    saidas = []
    for t, m in enumerate(modelos, 1):
        y = m.predizer(X_teste[i])
        valvula = 'A' if y == -1 else 'B'
        linha[f'T{t}'] = valvula
        saidas.append(valvula)
    
    linha['Válvula Destino'] = 'Válvula A (-1)' if saidas[0] == 'A' else 'Válvula B (+1)'
    tabela_q4.append(linha)

df_q4 = pd.DataFrame(tabela_q4)
display(df_q4.style.set_caption("Classificação das 15 Amostras de Sinais do ADALINE (adaline_teste.csv)"))

## Questão 5 — Explicação de Por Que os Pesos Permanecem Inalterados no ADALINE

**Pergunta:** Embora o número de épocas de cada treinamento realizado no item 2 seja diferente, explique por que então os valores dos pesos continuam praticamente inalterados.

**Resposta:**  
Diferente do Perceptron (que utiliza a saída discreta com a regra de Hebb), o ADALINE minimiza o **Erro Quadrático Médio (EQM)** utilizando a **Regra Delta (Gradiente Descendente)** sobre a combinação linear contínua $u = w^T x$.

1. A superfície de erro do EQM no ADALINE é uma superfície **quadrática estritamente convexa** (um paraboloide $N$-dimensional).
2. Por ser estritamente convexa, a superfície possui **UM ÚNICO MÍNIMO GLOBAL** (mínimo absoluto). Não existem mínimos locais.
3. A solução analítica dos pesos ideais $w^*$ é única e dada pela Equação Normal dos Mínimos Quadrados:  
   $$w^* = (X^T X)^{-1} X^T d$$
4. Independentemente dos pesos iniciais (ponto de partida na superfície de erro), o algoritmo do Gradiente Descendente sempre desce a rampa de erro em direção ao mesmo ponto de mínimo global $w^*$.

**Por que as épocas variam ligeiramente (890 vs 918 épocas)?**  
Diferentes pesos iniciais partem de distâncias e trajetórias distintas na superfície parabólica. Isso faz com que alguns treinamentos precisem de algumas poucas iterações a mais para que a variação do EQM atinja a precisão $\epsilon = 10^{-6}$ fixada como critério de parada.